In [1]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

def make_data(seed=0):
    return make_regression(
        n_samples=300,
        n_features=4,
        n_informative=4,
        noise=8.0,
        random_state=seed
    )

def add_bias(X):
    X = np.asarray(X)
    ones = np.ones((X.shape[0], 1), dtype=X.dtype)
    return np.hstack([ones, X])

def gd_fit(X, y, lr=0.1, n_iters=2000):
    Xb = add_bias(X)
    y = np.asarray(y)
    n, d_plus_1 = Xb.shape
    theta = np.zeros(d_plus_1, dtype=float)
    loss_history = []
    
    for _ in range(n_iters):
        resid = Xb @ theta - y
        loss = float(np.mean(resid ** 2))
        loss_history.append(loss)
        grad = (2.0 / n) * (Xb.T @ resid)
        theta = theta - lr * grad
        
    return theta, loss_history

def ridge_closed_form(X, y, alpha):
    Xb = add_bias(X)
    y = np.asarray(y)
    d_plus_1 = Xb.shape[1]
    I_prime = np.eye(d_plus_1)
    I_prime[0, 0] = 0.0
    theta = np.linalg.inv(Xb.T @ Xb + alpha * I_prime) @ (Xb.T @ y)
    return theta

def train_val_diagnostic(X, y, alpha, test_size=0.3, seed=0):
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, random_state=seed
    )
    theta = ridge_closed_form(X_train, y_train, alpha)
    pred_train = add_bias(X_train) @ theta
    pred_val = add_bias(X_val) @ theta
    train_mse = float(np.mean((y_train - pred_train) ** 2))
    val_mse = float(np.mean((y_val - pred_val) ** 2))
    return {"train_mse": train_mse, "val_mse": val_mse}


In [2]:
import numpy as np

rng = np.random.default_rng(11)
Xk = rng.standard_normal((300, 4))
true = np.array([5.0, 2.0, -3.0, 0.5, 1.5])
yk = Xk @ true[1:] + true[0] + rng.standard_normal(300) * 0.1
theta, hist = gd_fit(Xk, yk, lr=0.1, n_iters=3000)
print(np.round(theta, 2))
print(hist[0] > hist[-1], round(hist[-1], 3))
print(np.round(ridge_closed_form(Xk, yk, 0.0), 2))
print(train_val_diagnostic(Xk, yk, alpha=0.0))


[ 5.    1.99 -3.    0.51  1.51]
True 0.01
[ 5.    1.99 -3.    0.51  1.51]
{'train_mse': 0.009691134421986995, 'val_mse': 0.011178366604939573}
